# Exercise 2.3: Trip distribution (3 points): Person B

Report chapter: `report/chapters/06_trip_distribution.tex`

**Sub-questions**

1. Distance matrix (1 point)
2. Gravity model and resulting flows (1 point)
3. The ten largest flows (1 point)

**Reads**

- data/processed/zones.parquet (see the placeholder note below to start before Person A is done)

**Writes**

- data/processed/flows.parquet (origin_id, destination_id, distance, flow; zone order of zones.parquet)
- report/figures/distance_matrix.pdf
- report/figures/flow_matrix.pdf

**Note:** the toy example of the assignment is checked in this notebook.

## Setup and configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt

# ---- Study area ------------------------------------------------------------------------
DEPARTMENT = "14"            # INSEE code of Calvados (region Normandie)
CRS = "EPSG:2154"            # RGF93 / Lambert-93, metres; all geometries and distances

# ---- Paths -----------------------------------------------------------------------------
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir() and (p / "report").is_dir())
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
FIGURES_DIR = ROOT / "report" / "figures"

ZONES_PARQUET = DATA_PROCESSED / "zones.parquet"              # HAND-OFF from ex21_22
ZONES_PLACEHOLDER_MARKER = DATA_PROCESSED / "zones.PLACEHOLDER"
FLOWS_PARQUET = DATA_PROCESSED / "flows.parquet"              # ex23 -> ex31

# ---- Model parameters (assignment, Autumn 2026) -----------------------------------------
# Exercise 2.3, Table 2: rho_ij = exp(beta * d_ij + alpha), d_ij in metres.
# alpha is a constant factor exp(alpha) that cancels in the balancing (absorbed by p_i, a_j);
# TODO: confirm the symbols of the two Table 2 parameters in the assignment PDF.
FRICTION_BETA = -1.1e-4
FRICTION_ALPHA = -0.4
GRAVITY_MAX_ITER = 500
GRAVITY_TOL = 1e-10  # max relative change of p_i, a_j between two iterations

# ---- Hand-off file zones.parquet (ex21_22 writes, ex23-ex32 read) --------------------------------
# One row per municipality: municipality_id (str, unique), name (str), geometry (EPSG:2154),
# O (float >= 0, scaled trips O'_i), D (float >= 0, arriving trips D_j), with O.sum() == D.sum() == N.
# The row order fixes the zone order of all vectors and matrices in Person B's part.
ZONE_COLUMNS = ["municipality_id", "name", "geometry", "O", "D"]


def validate_zones(zones, rtol=1e-9):
    # Raise ValueError if the hand-off contract is violated
    missing = [c for c in ZONE_COLUMNS if c not in zones.columns]
    if missing:
        raise ValueError(f"zones is missing columns: {missing}")
    if zones.crs is None or not zones.crs.equals(CRS):
        raise ValueError(f"zones must have CRS {CRS}, got {zones.crs}")
    if zones["municipality_id"].duplicated().any():
        raise ValueError("municipality_id must be unique")
    if zones[["O", "D"]].isna().any().any() or (zones[["O", "D"]] < 0).any().any():
        raise ValueError("O and D must be non-negative and not NaN")
    if not np.isclose(zones["O"].sum(), zones["D"].sum(), rtol=rtol, atol=0.0):
        raise ValueError(f"O.sum() != D.sum(): {zones['O'].sum()} vs {zones['D'].sum()} (both must equal N)")


def read_zones():
    # Read and validate zones.parquet; warn if it is placeholder data
    import warnings
    if ZONES_PLACEHOLDER_MARKER.exists():
        warnings.warn("zones.parquet is PLACEHOLDER data (see data/processed/zones.PLACEHOLDER); "
                      "do not use it for final results.")
    zones = gpd.read_parquet(ZONES_PARQUET)
    validate_zones(zones)
    return zones

# ---- Figures ---------------------------------------------------------------------------
mpl.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 300, "font.size": 9, "axes.titlesize": 10,
    "axes.labelsize": 9, "legend.fontsize": 8, "pdf.fonttype": 42,
})


def style_map_axes(ax):
    # Common map look: equal aspect, no axes frame or ticks
    ax.set_aspect("equal")
    ax.set_axis_off()
    return ax


def save_figure(fig, name, preview_png=False):
    # Vector PDF for the report (report/figures/<name>.pdf, \includegraphics{<name>}), optional PNG preview
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    path = FIGURES_DIR / f"{name}.pdf"
    fig.savefig(path, format="pdf", bbox_inches="tight")
    if preview_png:
        fig.savefig(path.with_suffix(".png"), format="png", dpi=150, bbox_inches="tight")
    return path

**Starting before ex21_22 is done.** Until Person A writes `zones.parquet`, Person B can build PLACEHOLDER zones here
(O = population 15+ per municipality, D = URSSAF employment per municipality, both scaled to the same total
`N = 0.7 * min(sum(pop15+), sum(employment))`), write them to `zones.parquet` and create the marker file
`data/processed/zones.PLACEHOLDER`. `read_zones()` warns while the marker exists; ex21_22's final `write_zones(zones)` removes
it. Placeholder results must not go into the report.

### Gravity model (functions)

Friction (d in metres): `rho_ij = exp(beta * d_ij + alpha)`, beta = -1.1e-4, alpha = -0.4.
Weighting factors, initialised to 1 and iterated until they stabilise:
`p_i = O_i / sum_j a_j rho_ij`, `a_j = D_j / sum_i p_i rho_ij`.
Flows: `F_ij = p_i * a_j * rho_ij` (at convergence).

- A constant factor in rho (e.g. exp(alpha)) is absorbed by p and a and does not change F.
- The iteration only converges to the exact marginals if `sum(O) == sum(D)`; check this first.
- The toy example of the assignment (Tables 3-5) below checks the implementation.

In [ ]:
def centroids(zones):
    # Centroid coordinates as an (n, 2) array [x, y] in metres (projected CRS only)
    c = zones.geometry.centroid
    return np.column_stack([c.x.to_numpy(), c.y.to_numpy()])


def distance_matrix(xy):
    # Euclidean distance matrix in metres: d[i, j] = ||xy[i] - xy[j]||, symmetric, zero diagonal
    xy = np.asarray(xy, dtype=float)
    diff = xy[:, None, :] - xy[None, :, :]
    return np.sqrt((diff**2).sum(axis=-1))


def friction_matrix(distance, beta=FRICTION_BETA, alpha=FRICTION_ALPHA):
    # rho_ij = exp(beta * d_ij + alpha)
    # TODO: implement
    raise NotImplementedError


def balancing_factors(origins, destinations, friction, max_iter=GRAVITY_MAX_ITER, tol=GRAVITY_TOL):
    # Start from p = a = 1, alternate p_i = O_i / sum_j a_j rho_ij and a_j = D_j / sum_i p_i rho_ij
    # until the largest relative change of p and a is below tol or max_iter is reached.
    # Returns p, a, n_iter (report n_iter; warn if max_iter was hit).
    # TODO: implement
    raise NotImplementedError


def flow_matrix(origins, destinations, friction, p, a):
    # F_ij from the balancing factors; row sums ~ origins, column sums ~ destinations
    # TODO: implement
    raise NotImplementedError


def double_constrained_gravity(origins, destinations, distance, beta=FRICTION_BETA, alpha=FRICTION_ALPHA,
                               max_iter=GRAVITY_MAX_ITER, tol=GRAVITY_TOL):
    # Full model: friction -> balancing factors -> flow matrix (floats; round or floor only for display)
    # TODO: implement by combining the three functions above
    raise NotImplementedError

### Check: toy example of the assignment (Tables 3-5)

About D_1: the assignment prints D_1 = 9462 in Table 3, which is a typo (9462 is d_14). The value that matches the
column sum of Table 5 is D_1 = 420. Note that 420 + 2294 + 48377 + 6621 = 57712, whereas sum(O) = 57715 (Table 5 shows
rounded-down flows). With D_1 = 423 the totals are balanced and the floored flows match Table 5 exactly.
This cell fails (NotImplementedError) until the functions above are implemented.

In [ ]:
O_TOY = np.array([3396, 5442, 43196, 5681], dtype=float)
D_ASSIGNMENT = np.array([420, 2294, 48377, 6621], dtype=float)  # D_1 = 420, see above
D_BALANCED = np.array([423, 2294, 48377, 6621], dtype=float)    # sums to sum(O) = 57715
DISTANCE_TOY = np.array([
    [0, 4066, 6595, 9462],
    [4066, 0, 4340, 10299],
    [6595, 4340, 0, 7287],
    [9462, 10299, 7287, 0],
], dtype=float)
EXPECTED_FLOORED = np.array([  # Table 5 of the assignment (values rounded down)
    [74, 220, 2661, 439],
    [61, 445, 4415, 518],
    [247, 1463, 37659, 3825],
    [38, 163, 3640, 1837],
])

# Floored flows reproduce Table 5 within +-2 (D_1 = 420 as in the assignment)
flows = double_constrained_gravity(O_TOY, D_ASSIGNMENT, DISTANCE_TOY)
assert flows.shape == (4, 4)
assert np.all(np.abs(np.floor(flows) - EXPECTED_FLOORED) <= 2)

# With sum(D) == sum(O) (D_1 = 423) the floored flows match Table 5 exactly, and the marginals hold
flows = double_constrained_gravity(O_TOY, D_BALANCED, DISTANCE_TOY)
np.testing.assert_array_equal(np.floor(flows), EXPECTED_FLOORED)
np.testing.assert_allclose(flows.sum(axis=1), O_TOY, rtol=1e-6)
np.testing.assert_allclose(flows.sum(axis=0), D_BALANCED, rtol=1e-6)

# A constant factor exp(alpha) in the friction is absorbed by p_i and a_j
reference = double_constrained_gravity(O_TOY, D_BALANCED, DISTANCE_TOY, alpha=0.0)
for alpha in [-0.4, 1.3]:
    np.testing.assert_allclose(double_constrained_gravity(O_TOY, D_BALANCED, DISTANCE_TOY, alpha=alpha),
                               reference, rtol=1e-6)
print("toy example: all checks passed")

## 1. Distance matrix (1 point)

Centroid distance matrix in metres (`centroids`, `distance_matrix`, defined below), as a table or plot. Figure: `distance_matrix`.

In [ ]:
# TODO

## 2. Gravity model and resulting flows (1 point)

Friction `rho_ij = exp(beta d_ij + alpha)` (beta = -1.1e-4, alpha = -0.4), iterate `p_i`, `a_j` from 1 until they stabilise, compute `F_ij` (functions below). First make the toy-example check pass. Report the flows as a table or plot; write `flows.parquet`. Figure: `flow_matrix`.

In [ ]:
# TODO

## 3. The ten largest flows (1 point)

Document and name the 10 pairs of municipalities with the largest flows (state whether intra-zonal flows are included).

In [ ]:
# TODO

## Problems encountered

TODO: note every problem and how it was solved (partial points are given for documented problems); copy it into the `\paragraph{Problems encountered.}` of the report chapter.